# Lezione 57 — Valutazione offline del lab

> **Modulo:** capstone · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezioni 13 (metriche), 21 (retrieval), 37 (valutazione).
>
> Obiettivo pratico unico: produrre un **unico report** che valuta i tre
> componenti principali — classificatore, retrieval, estrazione relazioni — sul
> set di test, prima di mettere insieme la pipeline.

## Teoria essenziale

Prima di dichiarare "funziona" servono numeri, componente per componente, su dati
**mai visti** (il test set):

- **classificatore**: accuratezza (Lezione 13);
- **retrieval**: Precision@k (Lezione 21);
- **estrazione relazioni**: F1 a livello di relazione (Lezione 37).

Un report unico rende il progetto **verificabile** e fa da baseline per le
regressioni future.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path('..') / 'datasets' / 'processed'
TIPI = ["episodic", "semantic", "preference"]
_idx = {t: i for i, t in enumerate(TIPI)}
_train = pd.read_csv(proc / 'memory_train.csv')
_train = _train[_train['type'].isin(TIPI)].reset_index(drop=True)

def _tok(t):
    return re.findall(r"[a-zA-Z]+", str(t).lower())

# --- classificatore bag-of-words + softmax (Lezione 54) ---
_vocab = {}
for _t in _train['text']:
    for _w in _tok(_t):
        _vocab.setdefault(_w, len(_vocab))

def _bow(testo):
    v = np.zeros(len(_vocab))
    for w in _tok(testo):
        if w in _vocab:
            v[_vocab[w]] += 1.0
    return v

def _softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

_Xtr = np.vstack([_bow(t) for t in _train['text']])
_ytr = np.array([_idx[t] for t in _train['type']])
_W = np.zeros((len(_vocab), len(TIPI)))
_b = np.zeros(len(TIPI))
_Y = np.eye(len(TIPI))[_ytr]
for _ in range(300):
    _P = _softmax(_Xtr @ _W + _b)
    _W -= 0.5 * (_Xtr.T @ (_P - _Y) / len(_Xtr) + 1e-3 * _W)
    _b -= 0.5 * (_P - _Y).mean(axis=0)

def classifica_tipo(testo):
    return TIPI[int(_softmax(_bow(testo)[None, :] @ _W + _b).argmax())]

# --- rappresentazione: embedding, entita' (Lezione 55) ---
DIM = 48
def embed(testo):
    v = np.zeros(DIM)
    for w in _tok(testo):
        v[int.from_bytes(w.encode(), 'little') % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def estrai_entita(testo):
    return [w for w in re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo)) if w not in {"The", "A", "User"}]

# --- relazioni: fallback a regole (Lezione 56) ---
_VERBI = {"visited": "visited", "met": "met", "likes": "likes",
          "prefers": "prefers", "works": "works_at", "lives": "lives_in"}
def estrai_relazioni(testo):
    ent = estrai_entita(testo)
    for v_sup, v_norm in _VERBI.items():
        if v_sup in testo.lower() and len(ent) >= 2:
            return [{"source": ent[0], "type": v_norm, "target": ent[1]}]
    return []

# --- importanza composita (Lezione 25, versione compatta) ---
_FORTI = {"prefers", "likes", "dislikes", "hates", "loves", "always", "never", "important"}
def calcola_importanza(testo):
    # composita (Lezione 25): lunghezza + parole forti + numero di entita'
    parole = _tok(testo)
    forti = sum(1 for w in parole if w in _FORTI)
    lung = min(len(parole) / 15.0, 1.0)
    n_ent = len(estrai_entita(testo))
    imp = 0.30 * lung + 0.40 * min(forti, 1) + 0.20 * min(n_ent, 2) / 2 + 0.10
    return round(float(min(imp, 1.0)), 2)

In [2]:
# valuto sul TEST set (mai visto in addestramento)
test = pd.read_csv(proc / 'memory_test.csv')
test = test[test['type'].isin(TIPI)].reset_index(drop=True)

acc = float(np.mean([classifica_tipo(t) == y for t, y in zip(test['text'], test['type'])]))
print(f"classificatore - accuratezza test: {acc:.2f} (n={len(test)})")

classificatore - accuratezza test: 0.93 (n=14)


In [3]:
# retrieval: per alcune query, i primi k risultati sono dello STESSO tipo? (proxy P@k)
banca = _train.head(40).reset_index(drop=True)
E = np.vstack([embed(t) for t in banca['text']])

def precision_at_k(query_text, query_type, k=3):
    sim = E @ embed(query_text)
    top = np.argsort(sim)[::-1][:k]
    return float(np.mean([banca['type'].iloc[i] == query_type for i in top]))

pk = np.mean([precision_at_k(t, y) for t, y in zip(test['text'][:10], test['type'][:10])])
print(f"retrieval - Precision@3 medio (stesso tipo): {pk:.2f}")

retrieval - Precision@3 medio (stesso tipo): 0.80


In [4]:
# estrazione relazioni: F1 su un piccolo gold set etichettato a mano
GOLD = [
    ("Marco visited Glasgow.", {("Marco", "visited", "Glasgow")}),
    ("Elena works for Nordica.", {("Elena", "works_at", "Nordica")}),
    ("The user prefers tea.", set()),
]
tp = fp = fn = 0
for testo, gold in GOLD:
    pred = {(r["source"], r["type"], r["target"]) for r in estrai_relazioni(testo)}
    tp += len(pred & gold)
    fp += len(pred - gold)
    fn += len(gold - pred)
prec = tp / (tp + fp) if tp + fp else 1.0
rec = tp / (tp + fn) if tp + fn else 1.0
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
print(f"relazioni - P {prec:.2f} R {rec:.2f} F1 {f1:.2f}")

relazioni - P 1.00 R 1.00 F1 1.00


In [5]:
report = {"classificatore_acc": round(acc, 2),
          "retrieval_p@3": round(float(pk), 2),
          "relazioni_f1": round(f1, 2)}
# guardia di regressione: ogni componente sopra una soglia minima
assert report["classificatore_acc"] >= 0.6
assert report["relazioni_f1"] >= 0.6
print("REPORT DI VALUTAZIONE DEL LAB:", report)

REPORT DI VALUTAZIONE DEL LAB: {'classificatore_acc': 0.93, 'retrieval_p@3': 0.8, 'relazioni_f1': 1.0}


## Riepilogo (max 8 punti)

1. "Funziona" richiede **numeri**, componente per componente, su dati mai visti.
2. Classificatore: **accuratezza** (Lezione 13).
3. Retrieval: **Precision@k** (Lezione 21).
4. Relazioni: **F1** (Lezione 37).
5. Un report unico rende il progetto verificabile.
6. Le soglie diventano **guardie di regressione**.
7. Si valuta sul **test set**, non sul train.
8. E' il passo prima di assemblare la pipeline (Lezione 58).

## Quiz

1. Perche' valutare sul test set e non sul train?
2. Cosa misura Precision@3 in questo contesto?
3. A cosa serve una soglia minima nel report?

*(Risposte: 1. per stimare la generalizzazione su dati nuovi; 2. quanti dei primi
3 risultati simili sono dello stesso tipo della query; 3. come guardia di
regressione: un peggioramento sotto soglia fa fallire il controllo.)*